## EXAMPLE ON HOW TO USE RED BOOK INFORMATION ###



The **RED BOOK** is a table that contains information about drugs. Each drug is identified by
its *NDC* code (column **NDCNUM**). For each *NDC* code we have some "therapeutic information":

  * **Therapeutic Detail**: A 10-digit hierarchical 2008 RED BOOK ® code that
    categorizes drugs down to the generic ingredient level. This code is based
    on the American Hospital Formulary Service Classification Compilation
    (AHFSCC) Therapeutic Class.

 * **Therapeutic Class**: A 3-digit code that indicates the therapeutic/pharmacologic 
   category of the drug product. This is an aggregation over *Therapeutic Detail*, however
   the numerical codes between the therapeutic class and the therapeutic detail are
   not related, e.g. the therapeutic class "124" will not correspond to the therapeutic
   details that start with "124".

 * **Therapeutic Group**: A further aggregation of therapeutic class

The **claims** data has the following columns that are useful:

 * **NDCNUM**: This is a 11 digit number that uniquelly identifies each drug.
   This number is issued by the Food and Drug Administration.
   The first nine digits identify the manufacturer and product
   name. The last two digits identify the package size.
   **IMPORTANT**: Although this is an 11 digit number, it is stored as a string
   in the table

 * **THERCLS**: A 3 digit number with the therapeutic class of this drug (this is stored
   as an integer)
   
 * **THERGRP**: A 2 digit number with the therapeutic group of this drug.
   **IMPORTANT** This is also stored as a string, e.g. "01", "02" etc.

This information is usefull but is rather cryptic as it consists of some integer values
instead of some human readable descriptions. The claims table doesn't include *Therapeutic
Detail* either.

To solve these problems we can use the reference table **redbook**. This information has
the column **NDCNUM** which can be used to join with the claims table to retrieve
information about each drug. The *redbook* table has the columns **THERLS** and
**THERGRP** (just like the claims table does) but it also has the following extra
columns:
  * **THERDTL**: Therapeutic detail (an 10 digit number)
  * **THRDTDS**: A human readable description of the Therapeutic Detail
  * **THERCLDS**: A human readable description of the Therapeutic Class
  * **THRGRDS**: A human readable description of the Therapeutuc Group

**IMPORTANT NOTE**: Normally one would expect that we have a clean and well defined
tree hierarchy, e.g. each therapeutic class belongs to a single therapeutic group, and
each therapeutic detail belongs to a single therapeutic class.
This is *mostly* true, however there is a single exception:
 * Therapeutic class 234 (Misc Therapeutic Agents, NEC) is associated with both therapeutic
  group "15" (Respiratory Tract Agents) and "29" (Misc Therapeutic Agents).

Also, in some cases the therapeutic detail is NULL.





In [1]:
# -----------------------------------------------------------------------------
# INITIALIZATION
# -----------------------------------------------------------------------------
import sys
print(f"Python version: {sys.version}")

import logging
import os
import shutil
import csv
import gzip
import re
import copy
import json
import pandas as pd
import numpy as np

import plotly.express as px
import plotly.graph_objects as go

import pyspark
import pyspark.sql.functions as F
import pyspark.sql.types as T
from pyspark.sql import SparkSession
from pyspark import SparkConf

# -----------------------------------------------------------------------------
# INITIALIZE LOGGING
# -----------------------------------------------------------------------------
f = '%(asctime)-15s %(levelname)-8s %(message)s'
logger = logging.getLogger(__name__)
logger.setLevel("DEBUG")
logging.basicConfig(format=f)

# -----------------------------------------------------------------------------
# start_spark
# -----------------------------------------------------------------------------
def start_spark(
    driver_memory="100g",
    storage_fraction=0.5,
    num_nodes='*',
):
    """Initialize spark

    Arguments:
        driver_memory: Maximum heap size for the Spark driver Java
            virtual machine.
        storage_fraction: Controls what portion of Spark's unified
            memory is reserved for storage (i.e., caching/persisting data
            and broadcast variables), as a fraction of the total
            execution + storage memory pool.
            If you cache/persist a lot of data, and you're evicting
            data too early, you might increase this value (e.g. 0.6 or 0.7).
            Conversely, if your job is shuffle-heavy and fails due to
            memory pressure, you might decrease it (e.g. 0.3).
        num_nodes: How many concurrent threads to use while running
            in "local mode" (i.e. in a single machine instead of a cluster).
            Use '*' to use all cores, or an integer > 0 for a specific
            number of threads.
    """

    conf = SparkConf().setAppName("My_Application")
    conf.set("spark.driver.memory", driver_memory)
    conf.set("spark.memory.storageFraction", str(storage_fraction))
    conf.setMaster(f"local[{num_nodes}]")

    spark = SparkSession.builder.config(conf=conf).getOrCreate()
    spark.sparkContext.setLogLevel('WARN')

    return spark

spark = start_spark(num_nodes=16)
print(f">>> Spark version: {spark.version}")
#spark.stop()


from IPython.core.magic import register_cell_magic

@register_cell_magic
def spark_sql(line, cell):
    result = spark.sql(cell)
    result.show(n=1000)
    


Python version: 3.11.0 (main, Jun 13 2025, 14:48:45) [Clang 16.0.0 (clang-1600.0.26.6)]


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/09/17 10:28:25 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/09/17 10:28:25 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/09/17 10:28:25 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


>>> Spark version: 4.0.0


In [2]:
# READ OUR DATA

# -- First read the claims table
parquet_file = f"/Users/Charles/DATA/ckd/ckd_claims"
df_claims = spark.read.format("parquet").load(parquet_file)
df_claims.createOrReplaceTempView("claims")
print(f">>> Claims data frame has {df_claims.count():,} rows")

# -- Then read the redbook table
parquet_file = f"/Users/Charles/DATA/ckd/redbook"
df_redbook = spark.read.format("parquet").load(parquet_file)
df_redbook.createOrReplaceTempView("redbook")
print(f">>> Redbook data frame has {df_redbook.count():,} rows")
df_redbook.show(20)


25/09/17 10:28:27 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


>>> Claims data frame has 253,070,875 rows
>>> Redbook data frame has 473,890 rows
+-------+--------------------+--------------------+-------+--------------------+--------------------+-------+-------+-------+-----+-------+-------------+------------+----------+------+--------------------+-------+-------+-------+-------+-------+-------+-----------+--------------------+---------+-------+--------------------+------------+--------------------+--------------------+-------+--------------------+-----------------+------+-------+------+-------+
|REACTDT|               ROADS|             MSTFMDS|MASTFRM|             THRDTDS|             THRGRDS|PKQTYCD|THERCLS|EXCDGDS|ROACD|ORGBKFG|      METSIZE|     DEACLDS|   THERDTL|PKSIZE|             MANFNME|MAINTIN|THERGRP|EXCLDRG|DEACTDT|PRODCAT|ORGBKCD|     NDCNUM|             PRODNME|  ORGBKDS|GENERID|             THRCLDS|     STRNGTH|             GNINDDS|             PRDCTDS|DEACLAS|              GENNME|          MAINTDS|ACTIND|SIGLSRC|GENIND|DESIDRG|
+

# Get the pilot cohort

In [3]:

# ── 1) Load the pilot cohort (just ENROLIDs) ──────────────────
df_pilot = (
    spark.read.parquet("../0916_2017_18_with_2017_cost.parquet")
         .select("ENROLID")
         .distinct()                # one row per enrollee
)

# ── 2) Pull the RX claims that belong to those enrollees ──────
df_rx_claims = (
    df_claims                      # ←  your full claims table
        .filter(F.col("CLAIM_TYPE") == "RX")
        .join(df_pilot, on="ENROLID", how="inner")   # inner keeps only pilot IDs
)

In [ ]:
# Get unique THRCLDS and their corresponding THERDTL (or other descriptive columns)
thrclds_map = (
    df_redbook
    .select("THERCLS", "THRCLDS")  # or another description column
    .dropna()
    .dropDuplicates()
    .toPandas()
    .set_index("THERCLS")["THRCLDS"]
    .to_dict()
)
original_cols = [
    'THRCLS_53', 'THRCLS_51', 'THRCLS_52', 'THRCLS_69', 'THRCLS_46',
    'THRCLS_47', 'THRCLS_172', 'THRCLS_174', 'THRCLS_181', 'THRCLS_60'
]
codes = [col.split("_")[1] for col in original_cols]
rename_dict = {}
for col in original_cols:
    code = col.split("_")[1]
    # Get the therapeutic name (fallback to code if not found)
    name = thrclds_map.get(int(code), code)
    rename_dict[col] = f"{name} (THRCLS_{code})"
    print(code, name)



53 Antihyperlipidemic Drugs, NEC
51 Cardiac, Beta Blockers
52 Cardiac, Calcium Channel
69 Psychother, Antidepressants
46 Cardiac Drugs, NEC
47 Cardiac, ACE Inhibitors
172 Antidiabetic Agents, Insulins
174 Antidiabetic Agents, Misc
181 Immunosuppressants, NEC
60 Analg/Antipyr, Opiate Agonists


In [11]:
from functools import reduce
df = (
    spark.read.parquet("../0916_2017_18_with_2017_cost.parquet"))
for old_col, new_col in rename_dict.items():
    df = df.withColumnRenamed(old_col, new_col)
df.columns

['ENROLID',
 'annual_cost_2017',
 'highcost_gt_50000_2017',
 'highcost_gt_75000_2017',
 'highcost_gt_100000_2017',
 'highcost_gt_200000_2017',
 'highcost_gt_300000_2017',
 'highcost_gt_400000_2017',
 'highcost_gt_500000_2017',
 'INCOME_LEVEL',
 'annual_cost_2018_deflated',
 'highcost_gt_50000',
 'highcost_gt_75000',
 'highcost_gt_100000',
 'highcost_gt_200000',
 'highcost_gt_300000',
 'highcost_gt_400000',
 'highcost_gt_500000',
 'AGEGRP',
 'SEX',
 'REGION',
 'has_Hypertension',
 'has_Type_2_Diabetes',
 'has_Anemia',
 'has_Hyperlipidemia',
 'has_Acute_Kidney_Failure',
 'has_Hyperparathyroidism',
 'has_Kidney_Transplant',
 'has_Vitamin_D_Deficiency',
 'has_Long-term_Drug_Therapy',
 'has_Hypothyroidism',
 'has_Sleep_Apnea',
 'stage_2017',
 'util_2017',
 'Antihyperlipidemic Drugs, NEC (THRCLS_53)',
 'Cardiac, Beta Blockers (THRCLS_51)',
 'Cardiac, Calcium Channel (THRCLS_52)',
 'Psychother, Antidepressants (THRCLS_69)',
 'Cardiac Drugs, NEC (THRCLS_46)',
 'Cardiac, ACE Inhibitors (THRCLS_

In [12]:

df.write.mode("overwrite").parquet("../0917_2017_18_with_2017_cost.parquet")

In [4]:
# JOIN OUR CLAIMS WITH THE RED BOOK TABLE
redbook_columns = ["NDCNUM", "THERDTL", "THRGRDS", "THRCLDS", "THRDTDS"]

df_claims_redbook = df_rx_claims.join(df_redbook.select(redbook_columns), "NDCNUM", "left")

# -- display some rows (only for drug claims)
df_claims_redbook.filter(F.col("CLAIM_TYPE") == "RX").show(100)

+-----------+----------+---------+-------+----+----+-----+-------+------+-------+-----+----+-------+---+-------+-------+-----+-----+------+----+----+----+-----+-------+-------+--------+-------+--------+--------+-------+----------+----+-----+-------+------+---+-------+----------+-------+-------+------+----+-------+-------+-------+--------+-------+------+-----+-------+------+-------+-------+-------+-------+------+-------+-------+----+---+--------+-------+-------+----+-----+------+-----------+-------+--------+-------+-------+-------+--------+------+-------+------+----+-------+-------+------+-------+------+-------+-------+----------+----------+--------------------+--------------------+--------------------+
|     NDCNUM|   ENROLID|   SEQNUM|VERSION| DX1| DX2|PROC1|PROCTYP|CASEID|DISDATE|DOBYR|YEAR|ADMDATE|AGE|CAP_SVC|    COB|COINS|COPAY|DEDUCT| DRG| DX3| DX4|DXVER|FACHDID|FACPROF|MHSACOVG| NETPAY|NTWKPROV|PAIDNTWK|    PAY|    PDDATE| PDX|PPROC|PROCMOD|PROVID|QTY|REVCODE|   SVCDATE|SVCSCAT|T

In [30]:
# SOME STATISTICS ABOUT THE THERAPEUTIC GROUPS & CLASSES

n = df_claims_redbook.filter(F.col("CLAIM_TYPE") == "RX").count()
print(f">>> We have {n:,} Rx claims")

n_groups = df_claims_redbook.select("THERGRP").distinct().count()
n_classes = df_claims_redbook.select("THERCLS").distinct().count()
n_details = df_claims_redbook.select("THERDTL").distinct().count()

print(f">>> Number of distinct therapeutic groups in our claims: {n_groups:,}")
print(f">>> Number of distinct therapeutic classes in our claims: {n_classes:,}")
print(f">>> Number of distinct therapeutic details in our claims: {n_details:,}")

# -- FIND TOP 10 THERAPEUTIC GROUPS
print(">>> Top 10 therapeutic groups")
df = (
    df_claims_redbook.filter(F.col("CLAIM_TYPE") == "RX")
    .groupBy("THERGRP", "THRGRDS")
    .agg(F.count("*").alias("nrows"))
)
#df.orderBy("nrows", ascending=False).show(10, truncate=False)

# -- FIND TOP 10 THERAPEUTIC CLASSES
print(">>> Top 10 therapeutic classes (and their groups)")
df = (
    df_claims_redbook.filter(F.col("CLAIM_TYPE") == "RX")
    .groupBy("THERCLS", "THRCLDS", "THERGRP", "THRGRDS")
    .agg(F.count("*").alias("nrows"))
)
df.orderBy("nrows", ascending=False).show(10, truncate=False)



>>> We have 4,918,255 Rx claims


>>> Number of distinct therapeutic groups in our claims: 31
>>> Number of distinct therapeutic classes in our claims: 221
>>> Number of distinct therapeutic details in our claims: 1,273
>>> Top 10 therapeutic groups
>>> Top 10 therapeutic classes (and their groups)


+-------+------------------------------+-------+--------------------------+------+
|THERCLS|THRCLDS                       |THERGRP|THRGRDS                   |nrows |
+-------+------------------------------+-------+--------------------------+------+
|53     |Antihyperlipidemic Drugs, NEC |07     |Cardiovascular Agents     |396390|
|51     |Cardiac, Beta Blockers        |07     |Cardiovascular Agents     |269415|
|52     |Cardiac, Calcium Channel      |07     |Cardiovascular Agents     |251671|
|69     |Psychother, Antidepressants   |08     |Central Nervous System    |210437|
|46     |Cardiac Drugs, NEC            |07     |Cardiovascular Agents     |201487|
|47     |Cardiac, ACE Inhibitors       |07     |Cardiovascular Agents     |193483|
|174    |Antidiabetic Agents, Misc     |20     |Hormones & Synthetic Subst|170757|
|172    |Antidiabetic Agents, Insulins |20     |Hormones & Synthetic Subst|167366|
|60     |Analg/Antipyr, Opiate Agonists|08     |Central Nervous System    |133790|
|181

In [32]:
top10_thercls = df.orderBy("nrows", ascending=False).select("THERCLS","nrows","THRCLDS").limit(10).toPandas()
# ── 0) Grab the class names as a Python list ───────────────────
top_classes = top10_thercls["THERCLS"].tolist()      # already a pandas DF

# ── 1) Keep only RX claims from pilot IDs *and* those classes ──
df_rx_top = (
    df_rx_claims                            # from earlier join
        .filter(F.col("THERCLS").isin(top_classes))
        .select("ENROLID", "THERCLS")          # no other cols needed
        .distinct()                            # one row per (ENROLID, class)
)

# ── 2) Pivot to wide form: 10 columns, 1/0 flags ───────────────
df_flags = (
    df_rx_top
        .withColumn("flag", F.lit(1))
        .groupBy("ENROLID")
        .pivot("THERCLS", top_classes)         # keeps column order stable
        .agg(F.max("flag"))
        .na.fill(0)                            # any missing->0
)


In [33]:
import re
from pyspark.sql import functions as F

def make_safe(code):
    """
    Turn a numeric or string code into a Spark-legal column label.
      • Cast to string
      • Replace dots/whitespace/punctuation with '_'               (12.34 → '12_34')
    """
    # Convert 12.0 → '12' so we don’t get trailing '.0'
    if isinstance(code, float) and code.is_integer():
        code = int(code)
    return re.sub(r'\W+', '_', str(code).strip())

# ── Rename the pivoted columns ---------------------------------------------
rename_exprs = (
    [F.col("ENROLID")] +
    [
        F.col(f"`{c}`").alias(f"THRCLS_{make_safe(c)}")
        for c in top_classes          # numeric codes taken straight from your list
    ]
)

df_flags_prefixed = df_flags.select(*rename_exprs)

# ── Join back to the pilot cohort ------------------------------------------
df_pilot_feat = (
    df_pilot
        .join(df_flags_prefixed, on="ENROLID", how="left")
        .na.fill(0)
)
df_pilot_feat.show(10)

+---------+---------+---------+---------+---------+---------+---------+----------+----------+---------+----------+
|  ENROLID|THRCLS_53|THRCLS_51|THRCLS_52|THRCLS_69|THRCLS_46|THRCLS_47|THRCLS_174|THRCLS_172|THRCLS_60|THRCLS_181|
+---------+---------+---------+---------+---------+---------+---------+----------+----------+---------+----------+
| 28940901|        1|        0|        0|        0|        1|        0|         0|         0|        0|         1|
|131734201|        0|        0|        1|        0|        0|        0|         0|         0|        0|         0|
|134882601|        0|        0|        1|        1|        0|        1|         0|         0|        0|         1|
|218615001|        0|        0|        0|        0|        1|        0|         1|         1|        0|         0|
|261011701|        1|        1|        1|        0|        1|        0|         1|         0|        0|         0|
|387195001|        1|        0|        1|        0|        0|        0|         

In [34]:
df_pilot_feat.select("ENROLID").distinct().count()

39851

In [35]:
df_full = spark.read.parquet("df_0714_2018_19_stage_pivot_diagnoses_cost_stratum.parquet")
assert df_full.count() == df_pilot_feat.count()

df_full = (
    df_full
      .join(df_pilot_feat, on="ENROLID", how="inner") # stage of CKD by quarter
)
df_full.write.mode("overwrite").parquet("df_0714_2018_19_with_THERCLS.parquet")

In [36]:
df_full.show(10)

+---------+------+---+------+-------+----------------+-------------------+----------+------------------+------------------------+-----------------------+---------------------+------------------------+--------------------------------+------------------+------+------+------+------+------+------+------+------+-----------------+-----------------+---------+---------+---------+---------+---------+---------+---------+---------+----------+----------+---------+----------+
|  ENROLID|AGEGRP|SEX|REGION|INDSTRY|has_Hypertension|has_Type_2_Diabetes|has_Anemia|has_Hyperlipidemia|has_Acute_Kidney_Failure|has_Hyperparathyroidism|has_Kidney_Transplant|has_Vitamin_D_Deficiency|has_Other_Long-term_Drug_Therapy|has_Hypothyroidism|2017Q1|2017Q2|2017Q3|2017Q4|2018Q1|2018Q2|2018Q3|2018Q4|cost_stratum_2017|cost_stratum_2018|util_2017|util_2018|THRCLS_53|THRCLS_51|THRCLS_52|THRCLS_69|THRCLS_46|THRCLS_47|THRCLS_174|THRCLS_172|THRCLS_60|THRCLS_181|
+---------+------+---+------+-------+----------------+----------

In [9]:
# -- FIND TOP 10 THERAPEUTIC DETAILS 
print(">>> Top 10 therapeutic details (and their classes and groups)")
df = (
    df_claims_redbook.filter(F.col("CLAIM_TYPE") == "RX")
    .groupBy("THERDTL", "THRDTDS", "THERCLS", "THRCLDS", "THERGRP", "THRGRDS")
    .agg(F.count("*").alias("nrows"))
)
df.orderBy("nrows", ascending=False).show(10, truncate=False)

+-------+------+--------------------+
|THERCLS| nrows|             THRCLDS|
+-------+------+--------------------+
|     52|160504|Cardiac, Calcium ...|
|     53|142884|Antihyperlipidemi...|
|     47|130733|Cardiac, ACE Inhi...|
|     51|106370|Cardiac, Beta Blo...|
|     46|104380|  Cardiac Drugs, NEC|
|    178| 86508|Thy/Antithy, Thyr...|
|    120| 82528|Diuretics, Loop D...|
|     51| 78487|Cardiac, Beta Blo...|
|    235| 72334|Antigout Agents, NEC|
|    166| 68831|Adrenals & Comb, NEC|
|    174| 60310|Antidiabetic Agen...|
|     68| 54958|Anticonvulsants, ...|
|     85| 51771|Diabetes Mell/Dia...|
|     60| 49211|Analg/Antipyr, Op...|
|     54| 45326|Hypotensive Agent...|
|    181| 45024|Immunosuppressant...|
|    172| 44558|Antidiabetic Agen...|
|    224| 43669|      Vitamin D, NEC|
|     53| 42894|Antihyperlipidemi...|
|    181| 42520|Immunosuppressant...|
+-------+------+--------------------+
only showing top 20 rows


In [ ]:
# SHOW THE PROBLEM WITH THERAPEUTIC CLASS 234 (IT BELONGS TO BOTH THERAPEUTIC GROUPS 15 AND 29)
q = """
SELECT THERCLS, THRCLDS, THERGRP, THRGRDS, THERDTL, THRDTDS, COUNT(*) AS nrows
FROM redbook
WHERE THERCLS=234
GROUP BY THERCLS, THRCLDS, THERGRP, THRGRDS, THERDTL, THRDTDS
ORDER BY THERGRP
"""
spark.sql(q).show(100, truncate=False)
